# LivePortrait live photos - birthday test set

**Author**: Konrad Jelen (kj) <br>
**Date**: 2026-10-24 <br>
**Model**: LivePortrait, code `KwaiVGI/LivePortrait@9b294b3`, weights `KlingTeam/LivePortrait@82a4fa6` <br>
**Photo set**: `henryk-urodziny-2026`, three working copies <br>

Renders short live-photo clips of three birthday photos of Henry with LivePortrait, the first model under test for about 3 seconds of subtle, natural motion with the face itself unchanged. Every photo is animated in two variants with the same 3-second idle driving clip, and every clip is timed and measured, so the Star Colonel can view the clips side by side and judge them; the notebook measures and does not rank.

LivePortrait describes a face by implicit 3D keypoints: canonical keypoints x_c,s, a head rotation R, an expression offset δ, a scale s and a translation t. With relative motion on, output frame i moves the photo's keypoints by the change of driving frame i against the driving clip's first frame:

$$x_{d,i} = s_s \frac{s_{d,i}}{s_{d,0}} \left( x_{c,s} R_{d,i} R_{d,0}^{\top} R_s + \delta_s + \delta_{d,i} - \delta_{d,0} \right) + \left( t_s + t_{d,i} - t_{d,0} \right)_{xy}$$

The subscript xy means that the translation moves the keypoints in x and y only: LivePortrait sets its z component to zero.

- **`all` variant** - the equation as written: expression, head rotation, scale and position follow the driving clip
- **`exp` variant** - R_d,i R_d,0ᵀ becomes the identity, the scale ratio becomes 1 and the translation stays t_s, so only δ (expression, including the eyes) follows the driving clip and the head holds still
- **Expression-friendly option** (`driving_option`, the command-line default) - LivePortrait multiplies each frame's change against output frame 0 by a factor m, the size ratio of the photo's keypoint hull to the keypoint hull of output frame 0: x_d,i ← x_s + m·(x_d,i − x_d,0). With relative motion and a still photo, output frame 0 has the photo's own keypoints (x_d,0 = x_s), so m = 1 and the option does not change the motion in these 6 clips
- **Stitching and paste-back** - stitching aligns the animated 512 x 512 face crop with the shoulders; paste-back blends the crop into the full-resolution photo through a fixed mask, so pixels outside the mask stay those of the photo

## Approach
1. **Pin** the RTX PRO 6000 by UUID - every time and memory figure must come from one known card
2. **Load** the three working copies and the driving clip and report size and colour profile - the clip inherits the photo's resolution, and the profile decides how its colours display
3. **Load** LivePortrait once - the load is excluded from the per-clip times
4. **Detect** the faces LivePortrait finds in each photo - it animates only the largest, so the pick in a multi-face photo has to be visible
5. **Render** 3 photos × 2 variants (`all`, `exp`) with relative motion, stitching and paste-back on, restoring finished clips from their checkpoints - 6 clips
6. **Measure** wall time, peak VRAM and the pixel change inside each detected face, and show every clip under its photo - the Star Colonel judges the clips himself

## Outputs
- `data/processed/henryk-urodziny-2026/liveportrait/<stem>--<variant>.mp4` - 6 clips at the photo's own resolution
- `<stem>--<variant>.md` sidecar per clip (source working copy, model and commit, driving clip, settings, GPU, seconds, peak VRAM) and `checkpoints/<stem>--<variant>.json` with the same record for restore on re-run
- `<stem>--faces.jpg` - detected face boxes, for each photo with more than one detected face
- In-notebook: working-copy table, face-detection table, results table, per-face motion table, and every clip under its photo, linked by path so no photo or video bytes are stored in the notebook

## GPU Selection

Pins the RTX PRO 6000 Blackwell Max-Q (nvidia-smi index 1, 96 GB, compute capability 12.0) by UUID before torch is imported, because CUDA reads the variables once, at the first CUDA call. The Configuration render below confirms the card by name.

In [ ]:
import os

# Deterministic ordering so a UUID/index always names the same physical card
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# RTX PRO 6000 Blackwell Max-Q, pinned by UUID from `nvidia-smi -L` - survives reboots and hardware changes
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-a44e514a-aa5b-2844-35f1-4be28e1921e5"

import torch  # CUDA reads the env vars once, here - never set them after this import

## Imports

All imports in one cell. The project package is imported after the GPU pin because its model modules import torch; LivePortrait itself is imported when the pipeline loads.

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import tempfile  # Scratch folder for the discarded warm-up render
import time  # Pipeline load timing
from pathlib import Path  # File path handling

# Notebook display
from IPython.display import Image, Markdown, Video, display  # Photos and clips shown by path, never embedded

# Rich console output
from rich import box  # Table border style
from rich import print as rprint  # Formatted console printing
from rich.progress import (  # Progress bar components
    BarColumn,
    MofNCompleteColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)
from rich.table import Table  # Tabular output

# Project package
from henryk_media.animation import common  # Clip paths, measurement, sidecars, checkpoints
from henryk_media.animation.liveportrait import LivePortrait, save_face_overlay  # LivePortrait wrapper
from henryk_media.config import DATA_DIR, EXTERNAL_DATA_DIR, INTERIM_DATA_DIR  # Project paths

## Reproducibility

Seeds Python, NumPy and PyTorch. The render is a feed-forward pass that draws no random numbers, so the seed is set by convention; cuDNN benchmark mode, which LivePortrait switches on when it is imported, may still pick different convolution algorithms between runs, so the lowest bits of the frames are not guaranteed to repeat.

In [ ]:
common.set_seed(42)

## Configuration

Inputs, LivePortrait settings and the two render variants. Settings not listed keep LivePortrait's command-line defaults (FP16, crop scale 2.3, detection threshold 0.15), so the clips match what `inference.py` would render apart from the output resolution: `source_max_dim=0` turns off the downscale to 1280 px that LivePortrait applies by default. The driving clip is the 3-second idle cut documented in `data/external/driving-clips/idle-3s.md`: three blinks, a closed-lip slight smile and a small head dip, with the onset of an eyebrow raise in its last 0.3 s.

In [ ]:
# Inputs
PHOTO_SET = "henryk-urodziny-2026"  # folder name under data/interim and data/processed
WORKING_COPIES = [  # PNG working copies with EXIF orientation applied; originals stay in data/external
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8840.png",
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8564.png",
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8732.png",
]
DRIVING_CLIP = EXTERNAL_DATA_DIR / "driving-clips" / "idle-3s.mp4"  # 3 s cut of LivePortrait's d9.mp4: blinks, slight smile

# Output
OUTPUT_DIR = common.output_dir(PHOTO_SET, "liveportrait")  # clips, sidecars, checkpoints/, face overlays

# LivePortrait settings (ArgumentConfig fields)
LIVEPORTRAIT_SETTINGS = {
    "flag_relative_motion": True,  # motion relative to the driving clip's first frame
    "flag_stitching": True,  # align the animated face crop with the shoulders
    "flag_pasteback": True,  # blend the animated crop back into the full photo
    "source_max_dim": 0,  # 0 = no downscale (default 1280 px), clips keep the photo's resolution
    "driving_option": "expression-friendly",  # command-line default; multiplier is 1 with relative motion and a still photo, so no effect here
    "flag_use_half_precision": True,  # FP16 inference, the command-line default
}

# Render variants: name = LivePortrait animation_region
VARIANTS = {
    "all": "expression and head pose follow the driving clip",
    "exp": "expression and eyes follow the driving clip, head held still",
}

# Device - resolved at runtime; GPU info shown in render below
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
gpu_props = torch.cuda.get_device_properties(0)
gpu_compute_cap = f"{gpu_props.major}.{gpu_props.minor}"
gpu_memory_gb = gpu_props.total_memory / 1e9

settings_lines = "\n".join(f"  {key}: [grey70]{value}[/grey70]" for key, value in LIVEPORTRAIT_SETTINGS.items())
variant_lines = "\n".join(f"  [cadet_blue]{name}[/cadet_blue]: {text}" for name, text in VARIANTS.items())
photo_lines = "\n".join(f"  [cadet_blue]{photo.name}[/cadet_blue]" for photo in WORKING_COPIES)
rprint(f'''[bold medium_purple]Configuration[/bold medium_purple]
[dim]{"─" * 40}[/dim]
[bold slate_blue1]Inputs[/bold slate_blue1]
  Working copies: [dim]{WORKING_COPIES[0].parent}[/dim]
{photo_lines}
  Driving clip: [dim]{DRIVING_CLIP}[/dim]

[bold slate_blue1]LivePortrait settings[/bold slate_blue1]
{settings_lines}

[bold slate_blue1]Variants[/bold slate_blue1] [dim](animation_region)[/dim]
{variant_lines}

[bold slate_blue1]Output[/bold slate_blue1]
  Clips: [dim]{OUTPUT_DIR}[/dim]
  Clip count: [dark_sea_green]{len(WORKING_COPIES) * len(VARIANTS)}[/dark_sea_green]

[bold]Device[/bold]
  Using: [green]{device}[/green]
  GPU: [cyan]{gpu_name}[/cyan]
  Compute cap: [yellow]{gpu_compute_cap}[/yellow]
  Memory: [yellow]{gpu_memory_gb:.1f} GB[/yellow]
''')

## Data Loading

Reads each working copy for its size and embedded colour profile, and the driving clip for its frame count and rate. LivePortrait keeps the photo's resolution and the driving clip's frame count and rate, so these values predict every clip's format.

In [ ]:
drive_width, drive_height, drive_frames, drive_fps = common.probe_clip(DRIVING_CLIP)
photo_table = Table(
    title="Working copies",
    caption=f"Driving clip {DRIVING_CLIP.name}: {drive_width} x {drive_height} px, "
    f"{drive_frames} frames at {drive_fps:g} fps ({drive_frames / drive_fps:.2f} s)",
    box=box.SIMPLE_HEAVY,
)
for column in ("Photo", "Width x height", "Colour profile"):
    photo_table.add_column(column, style="grey70" if column == "Photo" else None)
for photo in WORKING_COPIES:
    height, width = common.load_working_copy(photo).shape[:2]
    photo_table.add_row(
        photo.name, f"[light_sea_green]{width} x {height} px[/light_sea_green]", common.colour_profile(photo) or "none"
    )
rprint(photo_table)

## Model

Loads the LivePortrait humans pipeline once through its Python API: the appearance, motion, warping, generator and stitching networks in PyTorch, and the InsightFace detector and landmark model in onnxruntime on CUDA. The load time is reported here and excluded from the per-clip times.

The second cell renders one clip that is then discarded. LivePortrait switches on cuDNN benchmark mode, which times candidate convolution algorithms on the first call of every layer, so the first render in a fresh process carries a one-time cost. The warm-up takes that cost out of the six measured clips and reports it as its own number; it runs on every execution, also when all clips are restored from checkpoints.

In [ ]:
start = time.perf_counter()
lp = LivePortrait(**LIVEPORTRAIT_SETTINGS)
load_seconds = time.perf_counter() - start
rprint(
    f"[bold medium_purple]LivePortrait loaded[/bold medium_purple]\n"
    f"  Revision: [cadet_blue]{lp.revision}[/cadet_blue]\n"
    f"  Load time: [light_sea_green]{load_seconds:.1f} s[/light_sea_green]\n"
    f"  PyTorch VRAM after load: [light_sea_green]{torch.cuda.memory_allocated() / 1e9:.2f} GB[/light_sea_green]"
)

In [ ]:
# Discarded warm-up render: its clip, sidecar and checkpoint go to a scratch folder removed on exit
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(dir=OUTPUT_DIR) as scratch:
    warmup = lp.render(WORKING_COPIES[0], DRIVING_CLIP, "all", Path(scratch) / "warmup.mp4")
rprint(
    f"[bold medium_purple]Warm-up render[/bold medium_purple] [dim](discarded)[/dim]\n"
    f"  Clip: [cadet_blue]{WORKING_COPIES[0].stem}--all[/cadet_blue]\n"
    f"  Time: [light_sea_green]{warmup.seconds:.1f} s[/light_sea_green]\n"
    f"  Peak VRAM: [light_sea_green]{warmup.peak_vram_gb:.2f} GB[/light_sea_green]"
)

### Face detection

Runs LivePortrait's own face detector on every working copy with the pipeline's settings. LivePortrait sorts the faces from the largest box to the smallest and animates only face 1; every other face stays as it is in the photo.

In [ ]:
faces = {photo: lp.detect_faces(photo) for photo in WORKING_COPIES}

face_table = Table(title="Faces LivePortrait detects", box=box.SIMPLE_HEAVY)
for column in ("Photo", "Face", "Box x0, y0, x1, y1", "Box size", "Detector score", "Animated"):
    face_table.add_column(column)
for photo, detections in faces.items():
    for number, ((x0, y0, x1, y1), score) in enumerate(detections, start=1):
        face_table.add_row(
            f"[grey70]{photo.name}[/grey70]",
            str(number),
            f"{x0}, {y0}, {x1}, {y1}",
            f"[light_sea_green]{x1 - x0} x {y1 - y0} px[/light_sea_green]",
            f"[dark_sea_green]{score:.2f}[/dark_sea_green]",
            "[dark_sea_green]✓ yes[/dark_sea_green]" if number == 1 else "[dim]no[/dim]",
        )
    face_table.add_section()
rprint(face_table)

In [ ]:
# Face boxes drawn on a copy of each multi-face photo; shown by path so no photo bytes enter the notebook
for photo, detections in faces.items():
    if len(detections) < 2:
        continue
    overlay = common.clip_path(OUTPUT_DIR, photo, "faces").with_suffix(".jpg")
    save_face_overlay(photo, [face_box for face_box, _ in detections], overlay)
    display(Markdown(f"**{photo.name}** - {len(detections)} faces; box 1 (green, labelled `1 animated`) is the face LivePortrait animates, red boxes stay still"))
    display(Image(url=os.path.relpath(overlay), width=480))

**IMG_8564 is the three-face photo.** LivePortrait's detector returns four boxes for it: three with detector scores 0.83-0.88 and box widths of 166-211 px, and a fourth box of 78 x 57 px at the top edge of the photo (score 0.46). LivePortrait animates box 1 (x 412-623, y 273-546), the middle one of the three faces from left to right, because it sorts the boxes by area and takes the largest. Box 2 on the right has the higher detector score (0.88 against 0.83) but the smaller box, so it is not picked. The faces on the left (box 3) and on the right (box 2) stay as they are in the photo; the per-face pixel change in the Results section confirms this. The notebook does not establish whether box 4 is a face.

- **IMG_8840** - two boxes: the animated face 1 (335 x 428 px, score 0.75) and a 118 x 343 px box against the right edge of the photo (x 915-1033 of 1034 px, score 0.40), which stays still
- **IMG_8732** - one face (314 x 396 px, score 0.80), animated

## Rendering

Renders each working copy in each variant: 3 photos × 2 variants = 6 clips, all with the same driving clip. Each clip's time and peak VRAM are measured around LivePortrait's `execute` call, after the PyTorch peak counter is reset. A clip whose video and checkpoint already exist is restored from the checkpoint instead of rendered again, so an interrupted run resumes where it stopped.

In [ ]:
jobs = [(photo, variant) for photo in WORKING_COPIES for variant in VARIANTS]
results, restored = [], set()

with Progress(
    SpinnerColumn(),
    TextColumn("[steel_blue]{task.description}"),
    BarColumn(bar_width=40),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
    refresh_per_second=10,
) as progress:
    task = progress.add_task("Rendering clips", total=len(jobs))
    for photo, variant in jobs:
        clip = common.clip_path(OUTPUT_DIR, photo, variant)
        progress.update(task, description=f"Rendering {clip.name}")
        result = common.load_result(clip)
        if result is None:
            result = lp.render(photo, DRIVING_CLIP, variant, clip)
        else:
            restored.add(result.clip)
        results.append(result)
        progress.advance(task)
    # force completion (prevents N-1 issue)
    progress.update(task, completed=len(jobs), description="Rendered clips")
    progress.refresh()

## Save/Export

Each render already wrote its clip, sidecar and checkpoint. This cell lists the files in the output folder and checks the two LivePortrait by-products that must not land in `data/`: the `.pkl` motion template, which LivePortrait writes beside the driving clip, and the side-by-side `_concat.mp4` video.

In [ ]:
written = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
stray_pkl = sorted(DATA_DIR.rglob("*.pkl"))
stray_concat = sorted(DATA_DIR.rglob("*_concat*"))
file_lines = "\n".join(
    f"  [cadet_blue]{path.relative_to(OUTPUT_DIR)}[/cadet_blue] [dim]{path.stat().st_size / 1e6:.2f} MB[/dim]" for path in written
)
rprint(
    f"[bold medium_purple]Output folder[/bold medium_purple] [dim]{OUTPUT_DIR}[/dim]\n{file_lines}\n\n"
    f"[bold slate_blue1]By-products in data/[/bold slate_blue1]\n"
    f"  .pkl motion templates: [{'dark_sea_green' if not stray_pkl else 'indian_red'}]{len(stray_pkl)}[/]\n"
    f"  _concat videos: [{'dark_sea_green' if not stray_concat else 'indian_red'}]{len(stray_concat)}[/]"
)

## Results

Measurements per clip, the pixel change inside every detected face, and the clips themselves under their source photos.

- **Time** - wall seconds of LivePortrait's `execute` call: face crop of the photo, face crop and motion extraction of the 90 driving frames, 90 animated frames with stitching and paste-back, and writing two videos (the result and the concat video, which is then discarded)
- **Peak VRAM** - `torch.cuda.max_memory_allocated` during the clip, including the loaded weights; memory that onnxruntime holds for the face detector is not counted
- **Face change** - for each detected face box b, the mean absolute pixel difference against frame 0, averaged over frames 1-89, on the 0-255 scale; 0 means the box never changed:

$$\Delta_b = \frac{1}{89} \sum_{i=1}^{89} \operatorname{mean}_{p \in b} \left| I_i(p) - I_0(p) \right|$$

In [ ]:
result_table = Table(title="LivePortrait clips", box=box.SIMPLE_HEAVY)
for column in ("Clip", "Variant", "Seconds", "Peak VRAM", "Size", "Frames", "Duration", "Measured"):
    result_table.add_column(column, justify="left" if column in ("Clip", "Variant", "Measured") else "right")
for result in results:
    result_table.add_row(
        f"[grey70]{Path(result.clip).name}[/grey70]",
        result.variant,
        f"[light_sea_green]{result.seconds:.1f} s[/light_sea_green]",
        f"[light_sea_green]{result.peak_vram_gb:.2f} GB[/light_sea_green]",
        f"{result.width} x {result.height}",
        f"[dark_sea_green]{result.frames}[/dark_sea_green]",
        f"[light_sea_green]{result.frames / result.fps:.2f} s[/light_sea_green]",
        "[dim]checkpoint[/dim]" if result.clip in restored else "this run",
    )
rprint(result_table)

In [ ]:
motion_table = Table(title="Pixel change inside each detected face (mean |frame - frame 0|, 0-255)", box=box.SIMPLE_HEAVY)
for column in ("Clip", "Face", "Animated", "Change"):
    motion_table.add_column(column, justify="right" if column == "Change" else "left")
for (photo, variant), result in zip(jobs, results):
    clip = common.clip_path(OUTPUT_DIR, photo, variant)
    changes = common.region_motion(clip, [face_box for face_box, _ in faces[photo]])
    for number, change in enumerate(changes, start=1):
        motion_table.add_row(
            f"[grey70]{Path(result.clip).name}[/grey70]",
            str(number),
            "[dark_sea_green]✓ yes[/dark_sea_green]" if number == 1 else "[dim]no[/dim]",
            f"[dark_sea_green]{change:.2f}[/dark_sea_green]",
        )
    motion_table.add_section()
rprint(motion_table)

In [ ]:
# Photo and clips linked by path relative to the notebook: no image or video bytes are stored here
for photo in WORKING_COPIES:
    display(Markdown(f"### {photo.stem}"))
    display(Image(url=os.path.relpath(photo), width=400))
    for variant, text in VARIANTS.items():
        clip = common.clip_path(OUTPUT_DIR, photo, variant)
        display(Markdown(f"`{variant}` - {text}"))
        display(Video(os.path.relpath(clip), embed=False, width=400, html_attributes="controls loop muted"))

All six clips were rendered in this run, none restored from a checkpoint. Each clip has the photo's own resolution and the driving clip's 90 frames at 30 fps (3.00 s).

- **Time grows with the photo's pixel count** - 8.4-8.7 s for IMG_8564 (962 x 1278, 1.23 MP), 8.7-9.0 s for IMG_8840 (1034 x 1374, 1.42 MP) and 11.8-12.1 s for IMG_8732 (1500 x 2000, 3.00 MP). The ranges of the two smaller photos meet at 8.7 s, so only the 3 MP photo is clearly slower. The networks run on face crops of fixed size (256 x 256 in, 512 x 512 out), and LivePortrait builds its side-by-side concat video from three 512 x 512 panels, so that video is 1536 x 512 px for every photo. Only paste-back into the full photo and the encode of the result video run on the CPU at the photo's resolution for every frame, so these two steps are where the time grows. At 3 MP a clip takes about 12 s on this card
- **Peak VRAM is the same for all six clips** - 1.34 GB of PyTorch allocations, of which 0.54 GB are the loaded weights. Because the networks work on fixed-size crops, the photo's resolution does not change the peak; GPU memory does not limit this workload on the 96 GB card
- **The first render in a process is slower** - the discarded warm-up render of IMG_8840 `all` took 35.0 s and peaked at 4.78 GB, against 9.0 s and 1.34 GB for the same clip measured afterwards. The difference is one-time work on the first call of each layer, which includes cuDNN benchmark mode (switched on by LivePortrait) timing candidate convolution algorithms with extra workspace memory
- **The two variants take the same time** - `all` and `exp` differ by at most 0.4 s per photo, because they change only which keypoint terms follow the driving clip, not which networks run
- **Under `exp` the animated face changes 39-44 % as much as under `all`** - 7.21 against 16.49 (IMG_8840), 4.44 against 11.18 (IMG_8564), 5.63 against 14.42 (IMG_8732). `exp` holds rotation, scale and position, so only the expression offsets move pixels, while `all` also moves the whole head. The number measures how much the pixels inside the face box change; it does not measure how natural the motion looks
- **Faces that LivePortrait does not animate stay at 0.01-0.03**, the level of H.264 compression noise. Box 3 of IMG_8564 shows 0.26 (`all`) and 0.13 (`exp`); this is consistent with the edge of face 1's paste-back region, which spans about 2.3 times the face, reaching into box 3, but the notebook does not localise the change
- **Colour profile** - IMG_8840 and IMG_8564 carry a Display P3 ICC profile. LivePortrait reads pixel values with OpenCV, which ignores ICC profiles, and the clips are written as H.264 with no colour tags. A colour-managed viewer therefore shows these two photos with P3 colours and their clips as standard RGB, so a clip can look less saturated than the photo above it. IMG_8732 has no profile, so photo and clip are read the same way

## Conclusions

LivePortrait rendered all 6 clips without an error: 3 photos × 2 variants, each 90 frames at 30 fps (3.00 s) at the photo's own resolution, saved with a sidecar and a checkpoint. Which clips work as live photos is the Star Colonel's judgement from the clips above; the facts below are what the notebook measured.

- **Cost** - pipeline load 4.4 s; 8.4-12.1 s per clip on the warm pipeline, growing with the photo's pixel count; 35.0 s for the first render in a fresh process (one-time first-call work, including cuDNN autotuning)
- **Memory** - 1.34 GB peak PyTorch allocation for every clip, 4.78 GB during the first render; onnxruntime memory is not included
- **Which face moved** - exactly one face per photo. In the three-face photo IMG_8564, LivePortrait animated the middle face (the largest box); the left and right faces changed by 0.26 and 0.03 against 11.18 for the animated face in `all`. In IMG_8840, a second box at the right edge of the photo stayed still
- **Variants** - `exp` holds the head still, and the animated face changes 39-44 % as much as under `all`; both variants take the same time
- **Inputs to keep in mind when viewing** - the driving clip's last 0.3 s (frames 81-89) contain the onset of an eyebrow raise with pursed lips (see `data/external/driving-clips/idle-3s.md`), and with relative motion every clip follows it at its end; the two Display P3 photos are shown as standard RGB in their clips
- **By-products** - no `.pkl` motion template and no `_concat` video in `data/`; the warm-up clip was discarded with its scratch folder
- **Not covered** - one driving clip and LivePortrait's default crop and motion settings only; face-change values measure pixel change, not naturalness or identity preservation